In [1]:
import pandas
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

In [121]:
#<source>_<table>_<YYYYMMDD>_<HHMMSS>.<ext>

In [2]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Read_From_MinIO") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "12345678") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()


last_dt = "20260430"

# df = spark.read.parquet("s3a://bronze/crm/") \
#     .filter(f"dt > '{last_dt}'")

# đọc parquet từ Bronze
df_cust = spark.read \
    .option("header", True) \
    .parquet("s3a://bronze/crm/cust/dt=20260430/")\
   

df_cust.show()


print ("tổng số lượng dòng trong df_cust: ", df_cust.count())


26/05/03 07:13:56 WARN Utils: Your hostname, dustin-Victus-by-HP-Laptop-16-d0xxx resolves to a loopback address: 127.0.1.1; using 192.168.1.28 instead (on interface wlo1)
26/05/03 07:13:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/dustin/Desktop/DataEngineer/Deltalake/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/dustin/.ivy2/cache
The jars for the packages stored in: /home/dustin/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-31c777aa-243b-4a31-a3bf-384655bac0fb;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.901 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 171ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------

+------+----------+-------------+------------+------------------+--------+---------------+----------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2026-05-01|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|2026-05-01|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|2026-05-01|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|2026-05-01|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|2026-05-01|
| 11005|AW00011005|        Julio|        Ruiz|                 S|       M|     2025-10-06|2026-05-01|
| 11006|AW00011006|        Janet|     Alvarez|                 S|       F|     202

In [123]:
#check số lượng sau khi lọc id null
df_cust = df_cust.filter(col('cst_id').isNotNull())
print("tổng số lượng dòng trong df_cust sau khi lọc id null: ", df_cust.count())


tổng số lượng dòng trong df_cust sau khi lọc id null:  18490


In [124]:
# check số lượng customer key null
df_cust.filter(col('cst_key').isNull()).count()
df_cust = df_cust.filter(col('cst_key').isNotNull())
print("tổng số lượng dòng trong df_cust sau khi lọc cst_key null: ", df_cust.count())

tổng số lượng dòng trong df_cust sau khi lọc cst_key null:  18490


In [125]:
df_cust.show(2)

+------+----------+-------------+------------+------------------+--------+---------------+----------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2026-05-01|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|2026-05-01|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
only showing top 2 rows



In [126]:
windowSpec = Window.partitionBy("cst_key").orderBy(col('cst_create_date').desc())
df_cust.withColumn("rk",row_number().over(windowSpec)).filter(col("cst_key") == 'AW00029433').show()

+------+----------+-------------+------------+------------------+--------+---------------+----------+---+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date| rk|
+------+----------+-------------+------------+------------------+--------+---------------+----------+---+
| 29433|AW00029433|       Thomas|        King|                 M|       M|     2026-01-27|2026-05-01|  1|
| 29433|AW00029433|         NULL|        NULL|                 M|       M|     2026-01-25|2026-05-01|  2|
+------+----------+-------------+------------+------------------+--------+---------------+----------+---+



chỉ lọc lấy giá trị up sau ngày đó lên vì giá trị trớc có thể bị sai


In [127]:
df_cust = df_cust.withColumn('rk', row_number().over(windowSpec)).filter(col('rk') == 1).drop('rk')
print ("đếm lại số lượng dòng sau khi lọc xong ", df_cust.count())

đếm lại số lượng dòng sau khi lọc xong  18484


In [128]:
df_cust.show()

+------+----------+-------------+------------+------------------+--------+---------------+----------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2026-05-01|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|2026-05-01|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|2026-05-01|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|2026-05-01|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|2026-05-01|
| 11005|AW00011005|        Julio|        Ruiz|                 S|       M|     2025-10-06|2026-05-01|
| 11006|AW00011006|        Janet|     Alvarez|                 S|       F|     202

điền lại trạng thái maried and single


In [129]:
df_cust = df_cust.withColumn('cst_marital_status', when(trim(col('cst_marital_status')) == 'M', 'Married')\
                                    .when(trim(col('cst_marital_status')) == 'S', 'Single')\
                                    .otherwise('N/a'))
df_cust = df_cust.withColumn('cst_gndr', when(trim(col('cst_gndr')) == 'M', 'Male')\
                                .when(trim(col('cst_gndr')) == 'F', 'Female')\
                                .otherwise('N/a'))

df_cust = df_cust.withColumn('cst_create_date', to_date(col('cst_create_date'), 'yyyy-MM-dd'))

In [130]:
df_cust.show(5)

+------+----------+-------------+------------+------------------+--------+---------------+----------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
| 11000|AW00011000|          Jon|       Yang |           Married|    Male|     2025-10-06|2026-05-01|
| 11001|AW00011001|       Eugene|     Huang  |            Single|    Male|     2025-10-06|2026-05-01|
| 11002|AW00011002|        Ruben|      Torres|           Married|    Male|     2025-10-06|2026-05-01|
| 11003|AW00011003|      Christy|         Zhu|            Single|  Female|     2025-10-06|2026-05-01|
| 11004|AW00011004|    Elizabeth|     Johnson|            Single|  Female|     2025-10-06|2026-05-01|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
only showing top 5 rows



In [131]:
# check kiểu dữ liệu pare lại cho đúng
df_cust.printSchema()

root
 |-- cst_id: string (nullable = true)
 |-- cst_key: string (nullable = true)
 |-- cst_firstname: string (nullable = true)
 |-- cst_lastname: string (nullable = true)
 |-- cst_marital_status: string (nullable = false)
 |-- cst_gndr: string (nullable = false)
 |-- cst_create_date: date (nullable = true)
 |-- load_date: date (nullable = true)



In [132]:
# kq cuối cùng
df_cust.show()

+------+----------+-------------+------------+------------------+--------+---------------+----------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
| 11000|AW00011000|          Jon|       Yang |           Married|    Male|     2025-10-06|2026-05-01|
| 11001|AW00011001|       Eugene|     Huang  |            Single|    Male|     2025-10-06|2026-05-01|
| 11002|AW00011002|        Ruben|      Torres|           Married|    Male|     2025-10-06|2026-05-01|
| 11003|AW00011003|      Christy|         Zhu|            Single|  Female|     2025-10-06|2026-05-01|
| 11004|AW00011004|    Elizabeth|     Johnson|            Single|  Female|     2025-10-06|2026-05-01|
| 11005|AW00011005|        Julio|        Ruiz|            Single|    Male|     2025-10-06|2026-05-01|
| 11006|AW00011006|        Janet|     Alvarez|            Single|  Female|     202

# crm_prd


In [133]:
# đọc parquet từ Bronze
df_prd = spark.read \
    .option("header", True) \
    .parquet("s3a://bronze/crm/prd/dt=20260430/")

df_prd.show()
print ("tổng số lượng dòng trong df_prd: ", df_prd.count())

+------+----------------+--------------------+--------+--------+------------+----------+----------+
|prd_id|         prd_key|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_dt| load_date|
+------+----------------+--------------------+--------+--------+------------+----------+----------+
|   210|CO-RF-FR-R92B-58|HL Road Frame - B...|    NULL|      R |  2003-07-01|      NULL|2026-05-01|
|   211|CO-RF-FR-R92R-58|HL Road Frame - R...|    NULL|      R |  2003-07-01|      NULL|2026-05-01|
|   212| AC-HE-HL-U509-R|Sport-100 Helmet-...|      12|      S |  2011-07-01|2007-12-28|2026-05-01|
|   213| AC-HE-HL-U509-R|Sport-100 Helmet-...|      14|      S |  2012-07-01|2008-12-27|2026-05-01|
|   214| AC-HE-HL-U509-R|Sport-100 Helmet-...|      13|      S |  2013-07-01|      NULL|2026-05-01|
|   215|   AC-HE-HL-U509|Sport-100 Helmet-...|      12|      S |  2011-07-01|2007-12-28|2026-05-01|
|   216|   AC-HE-HL-U509|Sport-100 Helmet-...|      14|      S |  2012-07-01|2008-12-27|2026-05-01|


In [134]:
df_prd = df_prd.filter(col('prd_id').isNotNull())
print("tổng số lượng dòng trong df_prd sau khi lọc id null: ", df_prd.count())

tổng số lượng dòng trong df_prd sau khi lọc id null:  397


In [135]:
df_prd = df_prd.withColumn('prd_start_dt', to_date(col('prd_start_dt'), 'yyyy-MM-dd'))
df_prd = df_prd.withColumn('prd_end_dt', to_date(col('prd_end_dt'), 'yyyy-MM-dd'))

In [136]:
df_prd.filter(col('prd_key') == 'AC-HE-HL-U509-R').show(5)


+------+---------------+--------------------+--------+--------+------------+----------+----------+
|prd_id|        prd_key|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_dt| load_date|
+------+---------------+--------------------+--------+--------+------------+----------+----------+
|   212|AC-HE-HL-U509-R|Sport-100 Helmet-...|      12|      S |  2011-07-01|2007-12-28|2026-05-01|
|   213|AC-HE-HL-U509-R|Sport-100 Helmet-...|      14|      S |  2012-07-01|2008-12-27|2026-05-01|
|   214|AC-HE-HL-U509-R|Sport-100 Helmet-...|      13|      S |  2013-07-01|      NULL|2026-05-01|
+------+---------------+--------------------+--------+--------+------------+----------+----------+



In [137]:
windowSpec = Window.partitionBy(col('prd_key')).orderBy(col('prd_id'))

In [138]:
df_prd = df_prd.withColumn('prd_end_dt', lead('prd_start_dt',1).over(windowSpec))#.filter(col('prd_key') == 'AC-HE-HL-U509-R').show(5)

df_prd.show(5)

+------+-------------+--------------------+--------+--------+------------+----------+----------+
|prd_id|      prd_key|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_dt| load_date|
+------+-------------+--------------------+--------+--------+------------+----------+----------+
|   478|AC-BC-BC-M005|Mountain Bottle Cage|       4|      M |  2013-07-01|      NULL|2026-05-01|
|   479|AC-BC-BC-R205|    Road Bottle Cage|       3|      R |  2013-07-01|      NULL|2026-05-01|
|   477|AC-BC-WB-H098|Water Bottle - 30...|       2|      S |  2013-07-01|      NULL|2026-05-01|
|   483|AC-BR-RA-H123| Hitch Rack - 4-Bike|      45|      S |  2013-07-01|      NULL|2026-05-01|
|   486|AC-BS-ST-1401|All-Purpose Bike ...|      59|      M |  2013-07-01|      NULL|2026-05-01|
+------+-------------+--------------------+--------+--------+------------+----------+----------+
only showing top 5 rows



In [139]:
df_prd = df_prd.withColumn('prd_line', when(trim(col('prd_line')) == 'M', 'Mountain')\
                            .when(trim(col('prd_line')) == 'R', 'Road')\
                            .when(trim(col('prd_line')) == 'S', 'Other Sales')\
                            .when(trim(col('prd_line')) == 'T', 'Touring')\
                            .otherwise ('N/a'))

In [140]:
df_prd.show(2)

+------+-------------+--------------------+--------+--------+------------+----------+----------+
|prd_id|      prd_key|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_dt| load_date|
+------+-------------+--------------------+--------+--------+------------+----------+----------+
|   478|AC-BC-BC-M005|Mountain Bottle Cage|       4|Mountain|  2013-07-01|      NULL|2026-05-01|
|   479|AC-BC-BC-R205|    Road Bottle Cage|       3|    Road|  2013-07-01|      NULL|2026-05-01|
+------+-------------+--------------------+--------+--------+------------+----------+----------+
only showing top 2 rows



In [141]:
df_prd = df_prd.withColumn('prd_key', trim(col('prd_key'))) \
               .withColumn('prd_nm', trim(col('prd_nm')))
df_prd.show()

+------+----------------+--------------------+--------+-----------+------------+----------+----------+
|prd_id|         prd_key|              prd_nm|prd_cost|   prd_line|prd_start_dt|prd_end_dt| load_date|
+------+----------------+--------------------+--------+-----------+------------+----------+----------+
|   478|   AC-BC-BC-M005|Mountain Bottle Cage|       4|   Mountain|  2013-07-01|      NULL|2026-05-01|
|   479|   AC-BC-BC-R205|    Road Bottle Cage|       3|       Road|  2013-07-01|      NULL|2026-05-01|
|   477|   AC-BC-WB-H098|Water Bottle - 30...|       2|Other Sales|  2013-07-01|      NULL|2026-05-01|
|   483|   AC-BR-RA-H123| Hitch Rack - 4-Bike|      45|Other Sales|  2013-07-01|      NULL|2026-05-01|
|   486|   AC-BS-ST-1401|All-Purpose Bike ...|      59|   Mountain|  2013-07-01|      NULL|2026-05-01|
|   484|   AC-CL-CL-9009|Bike Wash - Disso...|       3|Other Sales|  2013-07-01|      NULL|2026-05-01|
|   485|   AC-FE-FE-6654|Fender Set - Moun...|       8|   Mountain|  2013

In [142]:
# check lại 
df_prd.filter(col('prd_key') == 'AC-HE-HL-U509-R').show(5)

+------+---------------+--------------------+--------+-----------+------------+----------+----------+
|prd_id|        prd_key|              prd_nm|prd_cost|   prd_line|prd_start_dt|prd_end_dt| load_date|
+------+---------------+--------------------+--------+-----------+------------+----------+----------+
|   212|AC-HE-HL-U509-R|Sport-100 Helmet-...|      12|Other Sales|  2011-07-01|2012-07-01|2026-05-01|
|   213|AC-HE-HL-U509-R|Sport-100 Helmet-...|      14|Other Sales|  2012-07-01|2013-07-01|2026-05-01|
|   214|AC-HE-HL-U509-R|Sport-100 Helmet-...|      13|Other Sales|  2013-07-01|      NULL|2026-05-01|
+------+---------------+--------------------+--------+-----------+------------+----------+----------+



# crm sales


In [143]:
# đọc parquet từ Bronze
df_sales = spark.read \
    .option("header", True) \
    .parquet("s3a://bronze/crm/sales/dt=20260430/")

df_sales.show()
print ("tổng số lượng dòng trong df_sales: ", df_sales.count())

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price| load_date|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|    SO43697| BK-R93R-62|      21768|    20101229|   20110105|  20110110|     3578|           1|     3578|2026-05-01|
|    SO43698| BK-M82S-44|      28389|    20101229|   20110105|  20110110|     3400|           1|     3400|2026-05-01|
|    SO43699| BK-M82S-44|      25863|    20101229|   20110105|  20110110|     3400|           1|     3400|2026-05-01|
|    SO43700| BK-R50B-62|      14501|    20101229|   20110105|  20110110|      699|           1|      699|2026-05-01|
|    SO43701| BK-M82S-44|      11003|    20101229|   20110105|  20110110|     3400|           1|     3400|2026-05-01|
|    SO43702| BK-R93R-44|      27645|    20101230|   201

In [144]:
# check kiểu dữ liệu lại cho chuẩn


df_sales = df_sales.withColumn(
    "sls_order_dt",
    when(
        (col("sls_order_dt").isNull()) |
        (length(col("sls_order_dt")) != 8) |
        (col("sls_order_dt") == 0),
        None
    ).otherwise(col("sls_order_dt"))
)

df_sales = df_sales.withColumn(
    "sls_ship_dt",
    when(
        (col("sls_ship_dt").isNull()) |
        (length(col("sls_ship_dt")) != 8) |
        (col("sls_ship_dt") == 0),
        None
    ).otherwise(col("sls_ship_dt"))
)

df_sales = df_sales.withColumn(
    "sls_due_dt",
    when(
        (col("sls_due_dt").isNull()) |
        (length(col("sls_due_dt")) != 8) |
        (col("sls_due_dt") == 0),
        None
    ).otherwise(col("sls_due_dt"))
)
df_sales = df_sales.withColumn('sls_quantity', abs(col('sls_quantity')))

In [145]:
# check 
df_sales.show(5)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price| load_date|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|    SO43697| BK-R93R-62|      21768|    20101229|   20110105|  20110110|     3578|         1.0|     3578|2026-05-01|
|    SO43698| BK-M82S-44|      28389|    20101229|   20110105|  20110110|     3400|         1.0|     3400|2026-05-01|
|    SO43699| BK-M82S-44|      25863|    20101229|   20110105|  20110110|     3400|         1.0|     3400|2026-05-01|
|    SO43700| BK-R50B-62|      14501|    20101229|   20110105|  20110110|      699|         1.0|      699|2026-05-01|
|    SO43701| BK-M82S-44|      11003|    20101229|   20110105|  20110110|     3400|         1.0|     3400|2026-05-01|
+-----------+-----------+-----------+------------+------

In [146]:
df_sales.filter(col('sls_price') != col('sls_sales') * col('sls_quantity')).show(20)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price| load_date|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+----------+
|    SO57804| BK-M38S-40|      16470|    20130511|   20130518|  20130523|      769|         1.0|     -769|2026-05-01|
|    SO57916|    TI-M602|      23024|    20130513|   20130520|  20130525|       30|         1.0|      -30|2026-05-01|
|    SO58326|    FE-6654|      12045|    20130520|   20130527|  20130601|       22|         1.0|      -22|2026-05-01|
|    SO58335|    TI-M823|      13326|    20130520|   20130527|  20130601|       35|         2.0|       35|2026-05-01|
|    SO58623| BK-R79Y-44|      17012|    20130525|   20130601|  20130606|     1701|         1.0|    -1701|2026-05-01|
|    SO58818|    TI-R092|      21387|    20130528|   201

In [147]:
# convert lại 

df_sales = df_sales.withColumn('sls_sales', abs(col('sls_price')) * col('sls_quantity'))
df_sales = df_sales.withColumn('sls_price', abs(col('sls_price')))


# check erp cat

In [148]:
# đọc parquet từ Bronze
df_cat = spark.read \
    .option("header", True) \
    .parquet("s3a://bronze/erp/CAT/dt=20260430/")

df_cat.show()
print ("tổng số lượng dòng trong df_cat: ", df_cat.count())

+-----+-----------+-----------------+-----------+----------+
|   ID|        CAT|           SUBCAT|MAINTENANCE| load_date|
+-----+-----------+-----------------+-----------+----------+
|AC_BR|Accessories|       Bike Racks|        Yes|2026-05-01|
|AC_BS|Accessories|      Bike Stands|         No|2026-05-01|
|AC_BC|Accessories|Bottles and Cages|         No|2026-05-01|
|AC_CL|Accessories|         Cleaners|        Yes|2026-05-01|
|AC_FE|Accessories|          Fenders|         No|2026-05-01|
|AC_HE|Accessories|          Helmets|        Yes|2026-05-01|
|AC_HP|Accessories|  Hydration Packs|         No|2026-05-01|
|AC_LI|Accessories|           Lights|        Yes|2026-05-01|
|AC_LO|Accessories|            Locks|        Yes|2026-05-01|
|AC_PA|Accessories|         Panniers|         No|2026-05-01|
|AC_PU|Accessories|            Pumps|        Yes|2026-05-01|
|AC_TT|Accessories|  Tires and Tubes|        Yes|2026-05-01|
|BI_MB|      Bikes|   Mountain Bikes|        Yes|2026-05-01|
|BI_RB|      Bikes|     

In [158]:
# check null id

df_cat = df_cat.withColumn('ID', col('ID').isNotNull())\
                .withColumn('CAT', upper(trim(col('CAT'))))\
                .withColumn('MAINTENANCE', upper(trim(col('MAINTENANCE'))))\
                .withColumn('ID', upper(trim(col('ID'))))




# check null id



# check erp cust



In [159]:
# đọc parquet từ Bronze
df_erp_cust = spark.read \
    .option("header", True) \
    .parquet("s3a://bronze/erp/cust/dt=20260430/")

df_erp_cust.show()
print ("tổng số lượng dòng trong df_erp_cust: ", df_erp_cust.count())

+-------------+----------+------+----------+
|          CID|     BDATE|   GEN| load_date|
+-------------+----------+------+----------+
|NASAW00011000|1971-10-06|  Male|2026-05-01|
|NASAW00011001|1976-05-10|  Male|2026-05-01|
|NASAW00011002|1971-02-09|  Male|2026-05-01|
|NASAW00011003|1973-08-14|Female|2026-05-01|
|NASAW00011004|1979-08-05|Female|2026-05-01|
|NASAW00011005|1976-08-01|  Male|2026-05-01|
|NASAW00011006|1976-12-02|Female|2026-05-01|
|NASAW00011007|1969-11-06|  Male|2026-05-01|
|NASAW00011008|1975-07-04|Female|2026-05-01|
|NASAW00011009|1969-09-29|  Male|2026-05-01|
|NASAW00011010|1969-08-05|Female|2026-05-01|
|NASAW00011011|1969-05-03|  Male|2026-05-01|
|NASAW00011012|1979-01-14|Female|2026-05-01|
|NASAW00011013|1979-08-03|  Male|2026-05-01|
|NASAW00011014|1973-11-06|Female|2026-05-01|
|NASAW00011015|1984-08-26|Female|2026-05-01|
|NASAW00011016|1984-10-25|  Male|2026-05-01|
|NASAW00011017|1949-12-24|Female|2026-05-01|
|NASAW00011018|1955-10-06|  Male|2026-05-01|
|NASAW0001

In [170]:
#check CID
# df_erp_cust = df_erp_cust.filter(col("CID").isNotNull())

df_erp_cust = df_erp_cust.withColumn('CID',col("CID").isNotNull())\
                        .withColumn('BDATE', to_date(col('BDATE'), 'yyyy-MM-dd'))\
                        .withColumn('GEN', upper(trim(col('GEN'))))\
                        .withColumn('load_date', to_date(col('load_date'), 'yyyy-MM-dd'))



df_erp_cust.show()

+----+----------+------+----------+
| CID|     BDATE|   GEN| load_date|
+----+----------+------+----------+
|true|1971-10-06|  MALE|2026-05-01|
|true|1976-05-10|  MALE|2026-05-01|
|true|1971-02-09|  MALE|2026-05-01|
|true|1973-08-14|FEMALE|2026-05-01|
|true|1979-08-05|FEMALE|2026-05-01|
|true|1976-08-01|  MALE|2026-05-01|
|true|1976-12-02|FEMALE|2026-05-01|
|true|1969-11-06|  MALE|2026-05-01|
|true|1975-07-04|FEMALE|2026-05-01|
|true|1969-09-29|  MALE|2026-05-01|
|true|1969-08-05|FEMALE|2026-05-01|
|true|1969-05-03|  MALE|2026-05-01|
|true|1979-01-14|FEMALE|2026-05-01|
|true|1979-08-03|  MALE|2026-05-01|
|true|1973-11-06|FEMALE|2026-05-01|
|true|1984-08-26|FEMALE|2026-05-01|
|true|1984-10-25|  MALE|2026-05-01|
|true|1949-12-24|FEMALE|2026-05-01|
|true|1955-10-06|  MALE|2026-05-01|
|true|1983-09-04|  MALE|2026-05-01|
+----+----------+------+----------+
only showing top 20 rows



# check erp loc


In [171]:
# đọc parquet từ Bronze
df_erp_loc = spark.read \
    .option("header", True) \
    .parquet("s3a://bronze/erp/LOC/dt=20260430/")

df_erp_loc.show()
print ("tổng số lượng dòng trong df_erp_loc: ", df_erp_loc.count())

+-----------+---------+----------+
|        CID|    CNTRY| load_date|
+-----------+---------+----------+
|AW-00011000|Australia|2026-05-01|
|AW-00011001|Australia|2026-05-01|
|AW-00011002|Australia|2026-05-01|
|AW-00011003|Australia|2026-05-01|
|AW-00011004|Australia|2026-05-01|
|AW-00011005|Australia|2026-05-01|
|AW-00011006|Australia|2026-05-01|
|AW-00011007|Australia|2026-05-01|
|AW-00011008|Australia|2026-05-01|
|AW-00011009|Australia|2026-05-01|
|AW-00011010|Australia|2026-05-01|
|AW-00011011|Australia|2026-05-01|
|AW-00011012|       US|2026-05-01|
|AW-00011013|       US|2026-05-01|
|AW-00011014|       US|2026-05-01|
|AW-00011015|       US|2026-05-01|
|AW-00011016|       US|2026-05-01|
|AW-00011017|Australia|2026-05-01|
|AW-00011018|Australia|2026-05-01|
|AW-00011019|   Canada|2026-05-01|
+-----------+---------+----------+
only showing top 20 rows

tổng số lượng dòng trong df_erp_loc:  18484


In [174]:
# check CID

df_erp_loc = df_erp_loc.filter(col('CID').isNotNull())

df_erp_loc = df_erp_loc.withColumn('CID', upper(trim(col('CID'))))\
                        .withColumn('CNTRY', upper(trim(col('CNTRY'))))\
                        .withColumn('load_date', to_date(col('load_date'),  'yyyy-MM-dd'))


df_erp_loc.show()

+-----------+---------+----------+
|        CID|    CNTRY| load_date|
+-----------+---------+----------+
|AW-00011000|AUSTRALIA|2026-05-01|
|AW-00011001|AUSTRALIA|2026-05-01|
|AW-00011002|AUSTRALIA|2026-05-01|
|AW-00011003|AUSTRALIA|2026-05-01|
|AW-00011004|AUSTRALIA|2026-05-01|
|AW-00011005|AUSTRALIA|2026-05-01|
|AW-00011006|AUSTRALIA|2026-05-01|
|AW-00011007|AUSTRALIA|2026-05-01|
|AW-00011008|AUSTRALIA|2026-05-01|
|AW-00011009|AUSTRALIA|2026-05-01|
|AW-00011010|AUSTRALIA|2026-05-01|
|AW-00011011|AUSTRALIA|2026-05-01|
|AW-00011012|       US|2026-05-01|
|AW-00011013|       US|2026-05-01|
|AW-00011014|       US|2026-05-01|
|AW-00011015|       US|2026-05-01|
|AW-00011016|       US|2026-05-01|
|AW-00011017|AUSTRALIA|2026-05-01|
|AW-00011018|AUSTRALIA|2026-05-01|
|AW-00011019|   CANADA|2026-05-01|
+-----------+---------+----------+
only showing top 20 rows

